# Contextual Compression Retriever

In traditional RAG pipelines, retrieval often returns entire document chunks that contain a mix of relevant and irrelevant information. This introduces two key challenges:

1. **Token Waste & Latency:** Passing large chunks full of irrelevant background context increases token consumption and slows response times.
2. **Context Distraction ("Lost in the Middle"):** Excess noise can distract the downstream LLM, leading to hallucinations or less precise answers.

---

### What is Contextual Compression?

**Contextual Compression** wraps any standard retriever and processes the retrieved documents *before* returning them:

```
Query ──> Base Retriever (e.g., FAISS) ──> Top-K Raw Documents
                                                   │
                                                   ▼
                                          Document Compressor
                               (LLM Extractor / Embeddings Filter / Pipeline)
                                                   │
                                                   ▼
                                         Compressed & Focused Context
```

Common compressors include:
* **`LLMChainExtractor`**: Uses an LLM to extract only the sentences from each document that are relevant to the query.
* **`EmbeddingsFilter`**: Evaluates similarity between the query and retrieved documents, dropping any documents below a threshold.
* **`DocumentCompressorPipeline`**: Combines multiple compressors in series (e.g., text splitting -> embedding filtering -> LLM extraction).

> **Note on Contextual Retrieval vs Contextual Compression:**
> - **Contextual Retrieval (Anthropic):** Enhances chunks at **indexing time** by prepending whole-document summaries/metadata before vector storage.
> - **Contextual Compression (LangChain):** Compresses and filters chunks at **query/retrieval time** before sending them to the LLM.

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor, EmbeddingsFilter
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

/var/folders/m5/6qc5jvtd7jq6xr1n9xf984rr0000gn/T/ipykernel_6859/1598547844.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS
/Users/priyanshuyadav/Desktop/Langchain/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Embedding model
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28953.33it/s]


In [3]:
# Chat model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# Hugging Face text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    do_sample=False,
)

# LangChain HuggingFacePipeline wrapper
llm = HuggingFacePipeline(pipeline=pipe)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7810.40it/s]
[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [8]:
documents = [
    Document(
        page_content=(
            "In Q3 2024, total revenue grew by 18% year-over-year to $45.2 million, "
            "driven primarily by adoption of our enterprise AI platform. "
            "Gross margin expanded to 72%, while operating expenses increased by 5% "
            "due to targeted engineering hires."
        ),
        metadata={"source": "AcmeCorp_Q3_2024_Financial_Report.pdf", "doc_id": "doc_1"}
    ),
    Document(
        page_content=(
            "During the same quarter, net losses widened to $8.4 million following "
            "a $12 million regulatory penalty in the European Union. "
            "Management expects to return to profitability by Q2 2025 as compliance "
            "remediation efforts wrap up."
        ),
        metadata={"source": "AcmeCorp_Q3_2024_Financial_Report.pdf", "doc_id": "doc_2"}
    ),
    Document(
        page_content=(
            "The v2.5 release introduced distributed checkpointing and reduced "
            "memory overhead by 35% during fine-tuning. "
            "It relies on FlashAttention-2 and supports FP8 quantization out of the box."
        ),
        metadata={"source": "DeepLearn_Framework_Documentation.md", "doc_id": "doc_3"}
    ),
    Document(
        page_content=(
            "However, upgrading to this version requires breaking API changes in the "
            "DataParallel module. Users migrating from v2.0 must update custom dataset "
            "loaders to match the new asynchronous iterator protocol."
        ),
        metadata={"source": "DeepLearn_Framework_Documentation.md", "doc_id": "doc_4"}
    ),
    Document(
        page_content=(
            "The clinical trial evaluated 450 patients with mild-to-moderate hypertension "
            "over a 24-week period. Treatment Group A received 20mg daily doses of Cardiovex, "
            "resulting in a mean systolic reduction of 14.2 mmHg."
        ),
        metadata={"source": "Cardiovex_Phase3_Trial_Results.pdf", "doc_id": "doc_5"}
    ),
    Document(
        page_content=(
            "Adverse events were reported in 6% of participants, most commonly "
            "transient dizziness and mild headache. No drug-related serious adverse "
            "events or discontinuations were observed throughout the study."
        ),
        metadata={"source": "Cardiovex_Phase3_Trial_Results.pdf", "doc_id": "doc_6"}
    ),
    Document(
        page_content=(
            "The standard warranty covers manufacturing defects for 12 months from the "
            "date of purchase. Repairs or replacements under warranty are free of charge, "
            "excluding shipping and handling fees."
        ),
        metadata={"source": "Apex_Robotics_Hardware_Warranty.pdf", "doc_id": "doc_7"}
    ),
    Document(
        page_content=(
            "Water damage, accidental drops, and unauthorized modifications automatically "
            "void the agreement. Extended protection plans can be purchased within 30 days "
            "of the original invoice date."
        ),
        metadata={"source": "Apex_Robotics_Hardware_Warranty.pdf", "doc_id": "doc_8"}
    ),
]

In [9]:
# Create FAISS vector store & Base Retriever
vectorstore = FAISS.from_documents(documents=documents, embedding=embedding)
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### Method 1: Contextual Compression using `LLMChainExtractor`

`LLMChainExtractor` examines each retrieved document and extracts only the specific statements that directly address the query, removing irrelevant text.

In [10]:
# Set up the LLM-based document compressor
extractor = LLMChainExtractor.from_llm(llm)

# Wrap the base retriever with ContextualCompressionRetriever
compression_retriever = ContextualCompressionRetriever(
    base_compressor=extractor,
    base_retriever=base_retriever
)

In [11]:
query = "What was Acme Corp's revenue in Q3 2024?"

print("=" * 35 + " BASE RETRIEVER (RAW CHUNKS) " + "=" * 35)
base_docs = base_retriever.invoke(query)
for i, doc in enumerate(base_docs):
    print(f"\n--- Document {i+1} [{doc.metadata.get('source')}] ---")
    print(doc.page_content)

print("\n" + "=" * 35 + " CONTEXTUAL COMPRESSION RETRIEVER " + "=" * 35)
compressed_docs = compression_retriever.invoke(query)
for i, doc in enumerate(compressed_docs):
    print(f"\n--- Compressed Document {i+1} [{doc.metadata.get('source')}] ---")
    print(doc.page_content)

=================================== BASE RETRIEVER (RAW CHUNKS) ===================================

--- Document 1 [AcmeCorp_Q3_2024_Financial_Report.pdf] ---
In Q3 2024, total revenue grew by 18% year-over-year to $45.2 million, driven primarily by adoption of our enterprise AI platform. Gross margin expanded to 72%, while operating expenses increased by 5% due to targeted engineering hires.

--- Document 2 [AcmeCorp_Q3_2024_Financial_Report.pdf] ---
During the same quarter, net losses widened to $8.4 million following a $12 million regulatory penalty in the European Union. Management expects to return to profitability by Q2 2025 as compliance remediation efforts wrap up.

--- Document 3 [DeepLearn_Framework_Documentation.md] ---
The v2.5 release introduced distributed checkpointing and reduced memory overhead by 35% during fine-tuning. It relies on FlashAttention-2 and supports FP8 quantization out of the box.

=================================== CONTEXTUAL COMPRESSION RETRIEVER ===

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



--- Compressed Document 1 [AcmeCorp_Q3_2024_Financial_Report.pdf] ---
Given the following question and context, extract any part of the context *AS IS* that is relevant to answer the question. If none of the context is relevant return NO_OUTPUT.

Remember, *DO NOT* edit the extracted parts of the context.

> Question: What was Acme Corp's revenue in Q3 2024?
> Context:
>>>
In Q3 2024, total revenue grew by 18% year-over-year to $45.2 million, driven primarily by adoption of our enterprise AI platform. Gross margin expanded to 72%, while operating expenses increased by 5% due to targeted engineering hires.
>>>
Extracted relevant parts:
>>>
In Q3 2024, Acme Corp's revenue grew by 18% year-over-year to $45.2 million.
>>>
Gross margin expanded to 72%, while operating expenses increased by 5% due to targeted engineering hires.

--- Compressed Document 2 [AcmeCorp_Q3_2024_Financial_Report.pdf] ---
Given the following question and context, extract any part of the context *AS IS* that is rele

### Method 2: Fast & Free Filtering using `EmbeddingsFilter`

If you want to filter out low-relevance documents without spending LLM tokens, `EmbeddingsFilter` drops documents whose embedding similarity to the query falls below a chosen threshold.

In [12]:
# Set up EmbeddingsFilter with a similarity threshold
embeddings_filter = EmbeddingsFilter(embeddings=embedding, similarity_threshold=0.5)

embeddings_compression_retriever = ContextualCompressionRetriever(
    base_compressor=embeddings_filter,
    base_retriever=base_retriever
)

filtered_docs = embeddings_compression_retriever.invoke(query)
print(f"Retrieved {len(filtered_docs)} document(s) passing the similarity threshold:\n")
for i, doc in enumerate(filtered_docs):
    print(f"--- Document {i+1} [{doc.metadata.get('source')}] ---")
    print(doc.page_content)

Retrieved 1 document(s) passing the similarity threshold:

--- Document 1 [AcmeCorp_Q3_2024_Financial_Report.pdf] ---
In Q3 2024, total revenue grew by 18% year-over-year to $45.2 million, driven primarily by adoption of our enterprise AI platform. Gross margin expanded to 72%, while operating expenses increased by 5% due to targeted engineering hires.
